# Notebook 03: Pix2Pix Conditional GAN for Virtual Staining

### Overview
In this notebook, we train a **Pix2Pix Conditional Generative Adversarial Network (cGAN)** to perform virtual staining of virus infection fluorescence from brightfield micrographs.

**Key Learning Objectives**:
1. Define the PatchGAN Discriminator ($D$) and U-Net Generator ($G$).
2. Combine Adversarial Loss ($L_{\text{cGAN}}$) with $L_1$ Reconstruction Loss.
3. Understand adversarial training dynamics and discriminator/generator loss tracking.
4. Save generator checkpoints and inspect high-contrast virtual staining outputs.

In [ ]:
# ==========================================================
# 1. Google Colab Setup & Environment Setup
# ==========================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[+] Google Colab detected!")
    !git clone https://github.com/ayakimovich/GenAI4BIA.git /content/GenAI4BIA
    %cd /content/GenAI4BIA/practical
    !pip install -r ../requirements.txt -q
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("."))

## 2. Initialize Models & Data Loaders

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.data import VIRVSDataset
from src.models import Pix2PixGenerator, PatchGANDiscriminator
from src.generate_mock_virvs_data import create_dataset_directory
from src.utils import plot_virtual_staining_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Using device: {device}")

data_dir = "./data/mock_virvs"
create_dataset_directory(data_dir, num_train=30, num_val=10, image_size=(256, 256))

# Pix2Pix GAN uses normalized range [-1, 1]
train_dataset = VIRVSDataset(root_dir=data_dir, split="train", normalize_range=(-1.0, 1.0))
val_dataset = VIRVSDataset(root_dir=data_dir, split="val", normalize_range=(-1.0, 1.0))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Instantiate Generator and Discriminator
netG = Pix2PixGenerator(in_channels=1, out_channels=1, features=[32, 64, 128, 256]).to(device)
netD = PatchGANDiscriminator(in_channels=2, features=[32, 64, 128, 256]).to(device)

optG = optim.Adam(netG.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = optim.Adam(netD.parameters(), lr=2e-4, betas=(0.5, 0.999))

criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()
lambda_l1 = 100.0

## 3. Adversarial Training Loop

In [ ]:
num_epochs = 15
g_losses, d_losses = [], []

print("[+] Training Pix2Pix Conditional GAN...")
for epoch in range(1, num_epochs + 1):
    netG.train()
    netD.train()
    running_g, running_d = 0.0, 0.0
    
    for batch in train_loader:
        y_bf = batch["brightfield"].to(device)
        x_real = batch["fluorescence"].to(device)
        
        # --------------------------------------------------
        # 1. Update Discriminator D
        # --------------------------------------------------
        optD.zero_grad()
        
        pred_real = netD(y_bf, x_real)
        loss_d_real = criterion_gan(pred_real, torch.ones_like(pred_real))
        
        x_fake = netG(y_bf)
        pred_fake = netD(y_bf, x_fake.detach())
        loss_d_fake = criterion_gan(pred_fake, torch.zeros_like(pred_fake))
        
        loss_D = (loss_d_real + loss_d_fake) * 0.5
        loss_D.backward()
        optD.step()
        
        # --------------------------------------------------
        # 2. Update Generator G
        # --------------------------------------------------
        optG.zero_grad()
        pred_fake_g = netD(y_bf, x_fake)
        loss_g_gan = criterion_gan(pred_fake_g, torch.ones_like(pred_fake_g))
        loss_g_l1 = criterion_l1(x_fake, x_real)
        
        loss_G = loss_g_gan + lambda_l1 * loss_g_l1
        loss_G.backward()
        optG.step()
        
        running_d += loss_D.item() * y_bf.size(0)
        running_g += loss_G.item() * y_bf.size(0)
        
    epoch_d = running_d / len(train_dataset)
    epoch_g = running_g / len(train_dataset)
    d_losses.append(epoch_d)
    g_losses.append(epoch_g)
    
    if epoch % 3 == 0 or epoch == num_epochs:
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] - Loss D: {epoch_d:.4f} | Loss G: {epoch_g:.4f}")

# Save trained Pix2Pix generator checkpoint
torch.save(netG.state_dict(), "./pix2pix_generator_virvs.pth")
print("[+] Model saved to pix2pix_generator_virvs.pth")

## 4. Plot Loss Curves & Visual Inspection

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), g_losses, label="Generator Loss", fontweight="bold")
plt.plot(range(1, num_epochs + 1), d_losses, label="Discriminator Loss", linestyle="--", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pix2Pix Conditional GAN Training Dynamics", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Visual comparison on validation sample
netG.eval()
val_batch = next(iter(val_loader))
with torch.no_grad():
    bf_sample = val_batch["brightfield"][0:1].to(device)
    gt_sample = val_batch["fluorescence"][0:1].to(device)
    pred_sample = netG(bf_sample)
    
    bf_vis = (bf_sample[0] + 1.0) / 2.0
    gt_vis = (gt_sample[0] + 1.0) / 2.0
    pred_vis = (pred_sample[0] + 1.0) / 2.0

plot_virtual_staining_comparison(
    bf_vis, gt_vis, pred_pix2pix=pred_vis,
    title="Pix2Pix Virtual Staining Prediction"
)